Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# 2. Consumer: stream the Iceberg table into ADW with MERGE

This notebook reads the Iceberg table that notebook 1 writes, as a stream, and keeps the ADW table `ORDERS` up to date with the latest state of each order. Start it after notebook 1.

```
 Iceberg table           readStream             foreachBatch
 order_events  -------> format("iceberg") ----> AdwSink.upsert_batch
 (notebook 1)                                     |  executors: array inserts into ORDERS_STG
                                                  |  driver:    one MERGE into ORDERS
                                                  |             + a row in STREAM_BATCH_LOG
 ADW also reads the Iceberg table in place:       v
 external table ORDER_EVENTS_ICEBERG          ADW: ORDERS, ORDERS_STG, STREAM_BATCH_LOG
```

For every micro-batch, `AdwSink.upsert_batch` in `adw_sink.py`:

1. Skips the batch if `STREAM_BATCH_LOG` already has it for this query -- its name plus the id Spark keeps in its checkpoint. After a restart, Spark replays the last micro-batch if it had not been marked complete; a query started on a new checkpoint gets a new id, so its batches are never mistaken for old ones.
2. Keeps only the newest event per order, so the MERGE never sees one key twice (ORA-30926) and fewer rows travel.
3. Has the executors insert the rows into a permanent staging table in parallel, with python-oracledb array inserts.
4. Runs one MERGE from staging into `ORDERS`, logs the batch and clears its staging rows, in one transaction. The MERGE applies an event only if it is newer than the one the row already reflects, deletes the order for an event with `op = 'D'`, and inserts new orders.

Per batch, this costs two short connections from the driver and one per partition from the executors; no DDL. The built-in `aidataplatform` connector also has a MERGE mode, but it creates and drops a staging table on every call: loading the same 2,000 rows took 13.2 s through the connector against 2.2 s here.

## Configuration

In [ ]:
# ---- shared configuration: keep these values identical in all three notebooks ---------------
CATALOG = "default"
SCHEMA = "iceberg_streaming_demo"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.order_events"

# The Iceberg table lives in your bucket as <WAREHOUSE>/<schema>/<table>. ADW's Hadoop catalog
# support finds a table by that layout (notebook 2 creates the ADW external table).
WAREHOUSE = "oci://<bucket>@<namespace>/iceberg-streaming/warehouse"
TABLE_LOCATION = f"{WAREHOUSE}/{SCHEMA}/order_events"

# Checkpoints and stop files live on a Volume (here an external volume over a bucket folder).
STATE_DIR = "/Volumes/default/default/iceberg_checkpoints/streaming-from-iceberg-source"
CONTROL_DIR = f"{STATE_DIR}/control"

# Workspace folder holding this sample's helper modules, stream_control.py and adw_sink.py.
SAMPLE_DIR = "/Workspace/streaming-from-iceberg-source"

# ---- ADW connection (see the README, "One-time setup") ------------------------------------------
CREDENTIAL_NAME = "adw_iceberg_stream"         # Credential Store entry: keys user, password, wallet_password, tns
WALLET_DIR = "/Volumes/default/default/adw_wallets/<wallet-folder>"   # the unzipped ADW wallet
ADW_OS_CREDENTIAL = "OCI_ICEBERG_CRED"         # DBMS_CLOUD credential in ADW that can read the bucket
OCI_REGION = "us-ashburn-1"                    # region of the bucket
EXTERNAL_TABLE = "ORDER_EVENTS_ICEBERG"        # ADW external table over the Iceberg source table
CONSUMER_QUERY = "orders_to_adw"               # streaming query name, also the key in STREAM_BATCH_LOG
CONSUMER_CHECKPOINT = f"{STATE_DIR}/consumer"  # checkpoint of notebook 2's query

TRIGGER = "15 seconds"
RUN_MINUTES = 90

assert "<" not in WAREHOUSE, "Set WAREHOUSE to your bucket and namespace."
import os
assert os.path.isdir(os.path.dirname(STATE_DIR)), f"Create the Volume first (README): {os.path.dirname(STATE_DIR)}"

In [ ]:
import sys
if SAMPLE_DIR not in sys.path:
    sys.path.insert(0, SAMPLE_DIR)
import adw_sink
import stream_control

# The Credential Store values are never printed; AIDP also masks them in cell output.
adw = adw_sink.AdwConnection.from_credential_store(aidputils.secrets, CREDENTIAL_NAME, WALLET_DIR)
# ORDERS, ORDERS_STG and STREAM_BATCH_LOG. The checkpoint tells this query's batches apart from
# those of an earlier query of the same name, for example one started on a new checkpoint.
sink = adw_sink.AdwSink(spark, adw, query_name=CONSUMER_QUERY, checkpoint_location=CONSUMER_CHECKPOINT)
sink.ensure_tables()
print("Connected to ADW:", sink.fetch("SELECT sys_context('userenv', 'service_name') FROM dual")[0][0])

## Let ADW read the Iceberg table in place

`create_iceberg_external_table` calls `DBMS_CLOUD.CREATE_EXTERNAL_TABLE` with ADW's **Hadoop catalog** support. It needs the warehouse folder and `<schema>.<table>`, which is why notebook 1 put the table at `<WAREHOUSE>/<schema>/<table>`:

```
format => {"access_protocol": {"protocol_type": "iceberg", "protocol_config": {
             "iceberg_catalog_type": "hadoop",
             "iceberg_warehouse": "https://objectstorage.<region>.oraclecloud.com/n/<namespace>/b/<bucket>/o/iceberg-streaming/warehouse",
             "iceberg_table_path": "iceberg_streaming_demo.order_events"}}}
```

What we observed with this external table:

- It reads the **current snapshot on every query**, through the table's `version-hint.text`: new commits show up without any refresh, also with old metadata files being cleaned up.
- Compaction, snapshot expiry and copy-on-write updates and deletes are fine.
- Delete files written by merge-on-read changes make every query fail with ORA-29913, "Delete files not supported", until they are compacted away -- see notebook 3. The source table is copy-on-write for this reason.
- The column list is fixed when the external table is created. After `ADD COLUMN`, it keeps working without the new column; create it again with `replace=True` to see it.

In [ ]:
created = sink.create_iceberg_external_table(EXTERNAL_TABLE, ADW_OS_CREDENTIAL, TABLE_LOCATION, OCI_REGION)
print("created" if created else "already existed", EXTERNAL_TABLE)
print("events visible in ADW:", sink.fetch(f"SELECT COUNT(*), MAX(event_id) FROM {EXTERNAL_TABLE}")[0])

## Read options

| Option | Value here | Effect -- all observed on AIDP, Iceberg 1.8.x |
|---|---|---|
| `streaming-skip-overwrite-snapshots` | `true` | Without it the query **fails** at the first `overwrite` snapshot: any copy-on-write `UPDATE`, row-level `DELETE`, `MERGE` that updates, or `INSERT OVERWRITE`. With it, such snapshots are skipped -- **including the rows they add**. |
| `streaming-skip-delete-snapshots` | `true` | The same for `delete` snapshots: a `DELETE` that drops whole files, such as a whole partition, or a merge-on-read delete. |
| `streaming-max-files-per-micro-batch` | `200` | Caps a micro-batch, for example when catching up a backlog. |
| `streaming-max-rows-per-micro-batch` | not set | A row cap. On this Iceberg version it is a hard cap, and a data file with more rows than the cap stops the stream silently -- prefer the file cap. |
| `stream-from-timestamp` | not set | Where a *new* query starts, in epoch milliseconds. Without it a new query reads the table from its oldest snapshot. |

The two skip options let this pipeline survive the changes that notebook 3 makes. Skipped changes are not lost for good: notebook 3 shows how to apply them to ADW from Iceberg's changelog. In a pipeline where upstream jobs should never rewrite the table, leaving them unset is the safer choice: the stream stops loudly instead of drifting. See the README for the full table of options and caveats.

In [ ]:
READ_OPTIONS = {
    "streaming-skip-overwrite-snapshots": "true",
    "streaming-skip-delete-snapshots": "true",
    "streaming-max-files-per-micro-batch": "200",
}

## Start the stream

The cell stays busy while the stream runs; each micro-batch prints a line, and `STREAM_BATCH_LOG` in ADW records it. Stop the stream from notebook 3, or wait for `RUN_MINUTES`.

In [ ]:
reader = spark.readStream.format("iceberg")
for key, value in READ_OPTIONS.items():
    reader = reader.option(key, value)

query = (reader.load(SOURCE_TABLE)
         .writeStream
         .foreachBatch(sink.upsert_batch)
         .trigger(processingTime=TRIGGER)
         .option("checkpointLocation", CONSUMER_CHECKPOINT)
         .queryName(CONSUMER_QUERY)
         .start())

stream_control.run_until_stopped(query, CONTROL_DIR, "consumer", max_minutes=RUN_MINUTES)

## Check the result in ADW

Run this once the stream cell above has returned -- notebook 3's last section stops the stream after the consumer has caught up. Both tables are queryable from ADW: `ORDERS`, the native table this stream maintains, and `ORDER_EVENTS_ICEBERG`, the Iceberg source read in place. `compare_with_source` joins them in one ADW query: the latest event per order, computed from the Iceberg table, against `ORDERS`. Once the stream has caught up, every count except the first two should be 0.

In [ ]:
print(sink.compare_with_source(EXTERNAL_TABLE))
for row in sink.fetch(f"""
        SELECT COUNT(*) AS batches, SUM(rows_in) AS events, SUM(rows_merged) AS rows_merged,
               MIN(applied_at) AS first_batch, MAX(applied_at) AS last_batch
        FROM STREAM_BATCH_LOG WHERE query_name = :q""", q=CONSUMER_QUERY):
    print(row)

## Clean up -- optional

Drops the ADW tables this notebook created. Notebook 1 drops the Iceberg table.

In [ ]:
CLEAN_UP = False   # set to True to drop ORDERS, ORDERS_STG, STREAM_BATCH_LOG and the external table

if CLEAN_UP:
    for table in (EXTERNAL_TABLE, sink.target, sink.staging, sink.batch_log):
        with adw.connect() as conn:
            conn.cursor().execute(f"""BEGIN EXECUTE IMMEDIATE 'DROP TABLE {adw_sink.identifier(table)} PURGE';
                                      EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;""")
        print("dropped", table)